# 02 — Baseline Model: TF-IDF + Logistic Regression & DistilBERT

**Project:** Clinical NLP — Drug Review Sentiment Classification with XAI & UQ  
**Author:** Garcia Wa Nnam  

---

## Objective

Establish two baseline classifiers for drug review sentiment:

1. **Classical NLP baseline** — TF-IDF features + Logistic Regression (fast, interpretable, widely used in health informatics)
2. **Transformer baseline** — DistilBERT fine-tuned for binary classification (state-of-the-art contextual representations)

Both models are evaluated on the held-out test set using accuracy, precision, recall, F1-score, and ROC-AUC.

---

## Modelling Rationale

Starting with a classical baseline before moving to transformers is good ML engineering practice:  
it gives a performance floor, surfaces the marginal benefit of the more complex model,  
and ensures the transformer's added computational cost is justified.

> **Clinical framing:** In health technology assessment, a model must be demonstrably better than simpler alternatives to justify deployment. This baseline comparison mirrors that logic.

## 1. Imports and Setup

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os, time
import warnings
warnings.filterwarnings('ignore')

# Classical NLP
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight

# Transformers
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW

# Project utilities
from utils import (
    full_evaluation_report,
    plot_confusion_matrix,
    plot_roc_curve,
    set_seed,
    results_to_dataframe
)

set_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

## 2. Load Cleaned Data

In [ ]:
df_train = pd.read_csv('../data/train_clean.csv')
df_test  = pd.read_csv('../data/test_clean.csv')

# For faster iteration during development, subsample training set
# Remove / increase SAMPLE_N for full training run
SAMPLE_N = 20000
df_train_sample = df_train.sample(n=SAMPLE_N, random_state=42).reset_index(drop=True)

X_train = df_train_sample['review_clean'].tolist()
y_train = df_train_sample['label'].tolist()
X_test  = df_test['review_clean'].tolist()
y_test  = df_test['label'].tolist()

print(f'Training samples : {len(X_train):,}')
print(f'Test samples     : {len(X_test):,}')
print(f'Label distribution (train): {pd.Series(y_train).value_counts().to_dict()}')

---
## 3. Baseline A: TF-IDF + Logistic Regression

**Why start here?**  
TF-IDF + Logistic Regression is the workhorse of clinical text classification in production systems.  
It is fast, transparent, and provides probability estimates that can be calibrated.  
It also serves as a meaningful benchmark: if DistilBERT doesn't substantially outperform it,  
the simpler model is preferable on grounds of interpretability and resource cost.

In [ ]:
# Class weights to handle imbalance
classes = np.unique(y_train)
cw = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, cw))
print(f'Class weights: {class_weight_dict}')

tfidf_lr_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2),    # unigrams + bigrams
        sublinear_tf=True,     # log-scaled TF
        min_df=3,
        strip_accents='unicode'
    )),
    ('clf', LogisticRegression(
        C=1.0,
        class_weight='balanced',
        max_iter=1000,
        solver='lbfgs',
        random_state=42
    ))
])

t0 = time.time()
tfidf_lr_pipeline.fit(X_train, y_train)
print(f'Training time: {time.time()-t0:.1f}s')

In [ ]:
y_pred_lr   = tfidf_lr_pipeline.predict(X_test)
y_prob_lr   = tfidf_lr_pipeline.predict_proba(X_test)[:, 1]

full_evaluation_report(y_test, y_pred_lr, y_prob_lr, model_name='TF-IDF + Logistic Regression')
plot_confusion_matrix(y_test, y_pred_lr, model_name='TF-IDF + LR',
                      save_path='../outputs/cm_tfidf_lr.png')

---
## 4. Baseline B: DistilBERT Fine-Tuned

DistilBERT (Sanh et al., 2019) is a distilled version of BERT — 40% smaller and 60% faster  
while retaining ~97% of BERT's performance on GLUE benchmarks.  
It is well-suited to resource-constrained environments (e.g. NHS infrastructure).

**Fine-tuning strategy:**  
- Pre-trained `distilbert-base-uncased` from HuggingFace  
- 2 epochs, batch size 32, learning rate 2e-5 with linear warmup  
- Class-weighted cross-entropy loss to handle imbalance  
- Max sequence length 128 tokens (truncation justified in EDA notebook)

In [ ]:
# ── Dataset class ──────────────────────────────────────────────────────────
class DrugReviewDataset(Dataset):
    """
    PyTorch Dataset wrapper for drug review text and binary labels.
    Tokenisation is performed at construction time so DataLoader batches
    contain pre-computed input_ids and attention_masks.
    """
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding='max_length',
            max_length=max_length,
            return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels':         self.labels[idx]
        }

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
tokenizer  = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

# Use a smaller subsample for the transformer to keep runtime manageable
# For a full run, increase BERT_TRAIN_N
BERT_TRAIN_N = 8000
df_bert_train = df_train_sample.sample(n=BERT_TRAIN_N, random_state=42)

train_dataset = DrugReviewDataset(
    df_bert_train['review_clean'].tolist(),
    df_bert_train['label'].tolist(),
    tokenizer
)
test_dataset = DrugReviewDataset(
    X_test[:3000],  # subset for evaluation speed
    y_test[:3000],
    tokenizer
)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_loader)}, Test batches: {len(test_loader)}')

In [ ]:
# ── Model, optimiser, scheduler ────────────────────────────────────────────
model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(DEVICE)

# Class weights as tensor for weighted CE loss
cw_tensor = torch.tensor(list(class_weight_dict.values()), dtype=torch.float).to(DEVICE)

EPOCHS = 2
LR     = 2e-5
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

loss_fn = torch.nn.CrossEntropyLoss(weight=cw_tensor)
print(f'Model loaded. Total parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── Training loop ──────────────────────────────────────────────────────────
train_losses = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    t_epoch = time.time()

    for batch_idx, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = loss_fn(outputs.logits, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        if (batch_idx + 1) % 50 == 0:
            print(f'  Epoch {epoch+1} | Batch {batch_idx+1}/{len(train_loader)} '
                  f'| Loss: {loss.item():.4f}')

    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f'Epoch {epoch+1}/{EPOCHS} complete. Avg Loss: {avg_loss:.4f} '
          f'({time.time()-t_epoch:.0f}s)')

# Save model
model.save_pretrained('../outputs/distilbert_baseline')
tokenizer.save_pretrained('../outputs/distilbert_baseline')
print('Model saved to outputs/distilbert_baseline/')

In [ ]:
# ── Evaluation ─────────────────────────────────────────────────────────────
model.eval()
all_preds, all_probs, all_labels = [], [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['labels']

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs   = torch.softmax(outputs.logits, dim=1)[:, 1].cpu().numpy()
        preds   = (probs >= 0.5).astype(int)

        all_preds.extend(preds)
        all_probs.extend(probs)
        all_labels.extend(labels.numpy())

y_pred_bert = np.array(all_preds)
y_prob_bert = np.array(all_probs)
y_true_bert = np.array(all_labels)

full_evaluation_report(y_true_bert, y_pred_bert, y_prob_bert, model_name='DistilBERT Baseline')
plot_confusion_matrix(y_true_bert, y_pred_bert, model_name='DistilBERT',
                      save_path='../outputs/cm_distilbert.png')

## 5. Side-by-Side Comparison

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score

# Align test sets (LR used full test, BERT used subset)
results = {
    'TF-IDF + LR': {
        'Accuracy': round(accuracy_score(y_test, y_pred_lr), 4),
        'F1 (Macro)': round(f1_score(y_test, y_pred_lr, average='macro'), 4),
        'F1 (Positive)': round(f1_score(y_test, y_pred_lr, pos_label=1, average='binary'), 4),
        'ROC-AUC': round(roc_auc_score(y_test, y_prob_lr), 4),
    },
    'DistilBERT': {
        'Accuracy': round(accuracy_score(y_true_bert, y_pred_bert), 4),
        'F1 (Macro)': round(f1_score(y_true_bert, y_pred_bert, average='macro'), 4),
        'F1 (Positive)': round(f1_score(y_true_bert, y_pred_bert, pos_label=1, average='binary'), 4),
        'ROC-AUC': round(roc_auc_score(y_true_bert, y_prob_bert), 4),
    }
}

results_df = results_to_dataframe(results)
print(results_df.to_string())
results_df.to_csv('../outputs/baseline_results.csv')

In [ ]:
# ROC comparison
plot_roc_curve(
    y_test,
    {'TF-IDF + LR': y_prob_lr,
     'DistilBERT (subset)': np.concatenate([y_prob_bert, np.full(len(y_test)-len(y_prob_bert), np.nan)])[:len(y_test)]},
    save_path='../outputs/roc_comparison.png'
)
# Note: for a fair comparison both should use the same test set
# The ROC for DistilBERT here is on the 3000-sample subset

---
## Summary

Both baselines have been trained and evaluated. Key observations:

- **TF-IDF + LR** is fast and competitive — a strong baseline for deployment scenarios where latency and interpretability matter
- **DistilBERT** captures contextual nuance that bag-of-words misses (e.g., negations, comparative language)
- Both models use class-weighted training to handle the ~65/35 imbalance
- Results are saved in `outputs/baseline_results.csv` for comparison in the ablation notebook

**Clinical relevance:** In a system analysing patient feedback at scale, the transformer's ability to handle context-dependent meaning (e.g., *"didn't help"* vs *"helped"*) could reduce false positives in safety monitoring applications.

→ Proceed to `03_ablation_xai_uq.ipynb`